### EpochAI Data Warehouse (EpochAI_DW) — Exploratory Data Analysis & Visualization

This notebook queries the SQL Server semantic layer (
w_models_enriched and 
w_annual_energy_trends) to analyze compute scaling, energy intensity, FinOps cost structures, and environmental transparency across frontier AI models:

1. **Scaling Laws & The Transparency Gap (2018–2026 YTD)**
2. **Energy Consumption vs. Compute Efficiency (Top 12 Models)**
3. **Training Budget Breakdown (CapEx vs. Power OpEx)**
4. **Predictive Scaling Regression & Hidden Demand**
5. **Geopolitical Grid Trade-Off & Site Selection**
6. **AI Governance & Reporting Risk**
7. **Competitive Landscape (Top 10 AI Labs)**
8. **Physics-Informed Energy Imputation & Prediction Uncertainty**
9. **AI Economics Pareto Frontier & 12-Month Inference TCO**

In [2]:
# Install dependencies if needed (run once in terminal):
# pip install pandas sqlalchemy pyodbc plotly nbformat kaleido

import urllib.parse
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set global Plotly styling
px.defaults.template = "plotly_white"

### 1. Connect to SQL Server (`EpochAI_DW`) & Load Analytical Views

In [3]:
# Update SERVER_NAME if you use a named instance (e.g., r'localhost\SQLEXPRESS')
SERVER_NAME = r"DESKTOP-9IKR9U6\SQLEXPRESS"
DATABASE_NAME = "EpochAI_DW"
DRIVER_NAME = "ODBC Driver 17 for SQL Server"  # Change to 'ODBC Driver 18 for SQL Server' if needed

params = urllib.parse.quote_plus(
    f"DRIVER={{{DRIVER_NAME}}};"
    f"SERVER={SERVER_NAME};"
    f"DATABASE={DATABASE_NAME};"
    f"Trusted_Connection=yes;"
    f"TrustServerCertificate=yes;"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# Load semantic views created in 03_views_analise_models.sql
df_master = pd.read_sql("SELECT * FROM vw_models_enriched", engine)
df_annual = pd.read_sql("SELECT * FROM vw_annual_energy_trends ORDER BY publication_year", engine)

print(f"Master View loaded: {df_master.shape[0]:,} models x {df_master.shape[1]} columns")
print(f"Models with valid energy & CO2 data: {df_master['facility_total_energy_mwh'].notna().sum()} ({df_master['facility_total_energy_mwh'].notna().mean()*100:.1f}%)")
df_master.head(3)

Master View loaded: 1,102 models x 45 columns
Models with valid energy & CO2 data: 174 (15.8%)


,model_id,model_name,organization,org_category,publication_date,publication_year,domain,accessibility,open_weights,frontier_model,...,facility_electricity_cost_usd,hardware_infra_cost_usd,GPQA_Diamond_Score,FrontierMath_Score,SWE_Bench_Score,MMLU_Score,WebDev_Arena_Elo,Composite_Capability_Index,usd_per_capability_point,co2_tons_per_capability_point
0,1,Qwen3-VL-235B-A22B,Alibaba,Industry,2025-09-23,2025,"Language,Vision,Multimodal,Video",Open weights (unrestricted),Yes,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Qwen3-Omni-30B-A3B,Alibaba,Industry,2025-09-22,2025,"Multimodal,Language,Vision,Speech,Video",Open weights (unrestricted),Yes,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,AgentFounder-30B,Alibaba,Industry,2025-09-16,2025,Language,Open weights (unrestricted),Yes,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 2. Scaling Laws & The Transparency Gap (2018–2026 YTD)

In [4]:
# Prepare clean annual metrics (2018-2026), preserving 2026* on the X-axis
years_df = pd.DataFrame({"publication_year": list(range(2018, 2027))})
agg_df = (
    df_master[df_master["publication_year"].between(2018, 2026)]
    .groupby("publication_year", as_index=False)
    .agg(
        total_models=("model_id", "count"),
        disclosed_models=("facility_total_energy_mwh", "count"),
        avg_log10_flop=("log10_flop", "mean"),
        median_co2_tons=("pue_adjusted_co2_tons", "median"),
    )
)

df_trend = (
    years_df.merge(agg_df, on="publication_year", how="left")
    .fillna({"total_models": 0, "disclosed_models": 0})
    .sort_values("publication_year")
)

df_trend["transparency_pct"] = np.where(
    df_trend["total_models"] > 0,
    (df_trend["disclosed_models"] / df_trend["total_models"]) * 100.0,
    0.0,
).round(1)

df_trend["year_label"] = df_trend["publication_year"].apply(
    lambda y: f"{int(y)}*" if int(y) == 2026 else str(int(y))
)

NAVY = "#1E3A5F"
CORAL = "#D95F43"
CORAL_LIGHT = "rgba(217, 95, 67, 0.45)"
TEAL = "#2A7B88"
TEAL_FILL = "rgba(42, 123, 136, 0.14)"

bar_colors = [CORAL_LIGHT if int(y) == 2026 else CORAL for y in df_trend["publication_year"]]
bar_labels = [
    (f"{v:,.0f} t" if v >= 10 else f"{v:.1f} t") if pd.notna(v) and v > 0 else "N/A"
    for v in df_trend["median_co2_tons"]
]

fig1 = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.18,
    row_heights=[0.64, 0.36],
    specs=[[{"secondary_y": True}], [{"secondary_y": False}]],
    subplot_titles=(
        "1. Typical Carbon Footprint vs. Computing Power per Model",
        "2. Industry Transparency: Share of Models Disclosing Energy Data (%)",
    ),
)

fig1.add_trace(
    go.Bar(
        x=df_trend["year_label"],
        y=df_trend["median_co2_tons"].fillna(0),
        name="Typical Carbon Footprint per Model (Tons CO₂)",
        marker=dict(color=bar_colors, line=dict(color=CORAL, width=1)),
        text=bar_labels,
        textposition="outside",
        cliponaxis=False,
        textfont=dict(size=11, color="#333333"),
    ),
    row=1,
    col=1,
    secondary_y=False,
)

fig1.add_trace(
    go.Scatter(
        x=df_trend["year_label"],
        y=df_trend["avg_log10_flop"],
        name="Average Training Compute (Log₁₀ FLOP)",
        mode="lines+markers",
        line=dict(color=NAVY, width=3),
        marker=dict(size=8, color=NAVY),
    ),
    row=1,
    col=1,
    secondary_y=True,
)

fig1.add_trace(
    go.Scatter(
        x=df_trend["year_label"],
        y=df_trend["transparency_pct"],
        name="Energy Data Disclosure Rate (%)",
        mode="lines+markers+text",
        fill="tozeroy",
        fillcolor=TEAL_FILL,
        line=dict(color=TEAL, width=2.5),
        marker=dict(size=7, color=TEAL),
        text=df_trend["transparency_pct"].apply(lambda p: f"{p:.0f}%"),
        textposition="top center",
        cliponaxis=False,
        textfont=dict(size=11, color=TEAL),
    ),
    row=2,
    col=1,
)

latest_pct = df_trend["transparency_pct"].iloc[-1]

fig1.update_layout(
    title=dict(
        text=(
            "<b>AI Models Are Getting Bigger and More Carbon-Intensive — Yet Less Transparent (2018–2026)</b><br>"
            f"<span style='font-size:13px; color:#555555;'>"
            f"While computing power and typical carbon emissions per model surged, public energy disclosure dropped to {latest_pct:.0f}%"
            "</span>"
        ),
        x=0.02,
        xanchor="left",
        y=0.96,
        yanchor="top",
    ),
    font=dict(family="Segoe UI, Arial, sans-serif", size=12, color="#2C3E50"),
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.08,
        xanchor="left",
        x=0.02,
        bgcolor="rgba(255,255,255,0.9)",
    ),
    height=750,
    margin=dict(t=165, b=80, l=75, r=75),
)

fig1.update_xaxes(showgrid=False, linecolor="#CCCCCC")
fig1.update_yaxes(
    title_text="Typical Emissions per Model (Tons CO₂)",
    range=[0, max(1100, df_trend["median_co2_tons"].max() * 1.32)],
    gridcolor="#EEEEEE",
    row=1,
    col=1,
    secondary_y=False,
)
fig1.update_yaxes(
    title_text="Training Compute (Log₁₀ FLOP)",
    range=[16.5, 26.2],
    dtick=2,
    showgrid=False,
    row=1,
    col=1,
    secondary_y=True,
)
fig1.update_yaxes(
    title_text="Disclosure Rate (%)",
    range=[0, 48],
    gridcolor="#EEEEEE",
    row=2,
    col=1,
)
fig1.update_xaxes(
    title_text="Release Year (*2026 includes data through November — year in progress)",
    row=2,
    col=1,
)

for annotation in fig1["layout"]["annotations"]:
    annotation["font"] = dict(size=13, color="#1E3A5F")
    annotation["x"] = 0.02
    annotation["xanchor"] = "left"
    annotation["yshift"] = 12

fig1.show()
fig1.write_image("../assets/charts/chart1_scaling_transparency.png", width=1200, height=750, scale=1)

### 3. Energy Consumption vs. Compute Efficiency (Top 12 Models)

In [5]:
# Prepare Top 12 distinct modern frontier models (2022+) by facility-adjusted energy consumption
df_energy_valid = df_master[
    df_master["facility_total_energy_mwh"].notna()
    & (df_master["training_compute_flop"] > 0)
    & (df_master["publication_year"] >= 2022)
].copy()
df_energy_valid["mwh_per_yottaflop"] = (
    df_energy_valid["facility_total_energy_mwh"] / (df_energy_valid["training_compute_flop"] / 1e24)
)

# Detect text hardware column vs numeric hardware quantity
str_hw_cols = [
    c for c in df_energy_valid.columns
    if ("chip" in c.lower() or "hardware" in c.lower())
    and df_energy_valid[c].dtype == "object"
]
num_hw_cols = [
    c for c in df_energy_valid.columns
    if ("quantity" in c.lower() or "num_" in c.lower() or "hardware" in c.lower())
    and pd.api.types.is_numeric_dtype(df_energy_valid[c])
    and "cost" not in c.lower()
]

hw_text_col = str_hw_cols[0] if str_hw_cols else None
hw_qty_col = num_hw_cols[0] if num_hw_cols else None

def format_model_label(row):
    name = str(row["model_name"]).strip()
    chip = ""
    if hw_text_col and pd.notna(row[hw_text_col]):
        raw_chip = str(row[hw_text_col]).replace("NVIDIA ", "").replace("Google ", "").strip()
        if raw_chip and raw_chip not in ("Unknown", "None", "nan"):
            chip = raw_chip
    qty = ""
    if hw_qty_col and pd.notna(row[hw_qty_col]) and row[hw_qty_col] > 0:
        q = int(round(row[hw_qty_col]))
        qty = f"{q/1000:.0f}k" if q >= 10000 else f"{q:,}"
    
    if chip and qty:
        return f"{name} ({qty} {chip})"
    elif chip:
        return f"{name} ({chip})"
    elif qty:
        return f"{name} ({qty} chips)"
    return name

df_energy_valid["model_label"] = df_energy_valid.apply(format_model_label, axis=1)

# Consolidate duplicate minor revisions (e.g., GPT-4 Mar vs Jun 2023)
df_energy_valid["base_key"] = df_energy_valid["model_name"].str.replace(r"\s*\((Mar|Jun)\s*2023\)", "", regex=True)
df_energy_dedup = (
    df_energy_valid.sort_values("facility_total_energy_mwh", ascending=False)
    .drop_duplicates(subset=["base_key"], keep="first")
)

df_top_energy = (
    df_energy_dedup.head(12)
    .iloc[::-1]
    .copy()
)

# Minimalist editorial palette
NAVY_PRIMARY = "#1E3A5F"      # Top mega-clusters (>=65,000 MWh)
SLATE_SECONDARY = "#7A99B8"   # Standard frontier runs
TEAL_EFFICIENT = "#124B24"    # Better than median intensity
CORAL_INTENSE = "#81A884"     # Higher than median intensity

energy_colors = [
    NAVY_PRIMARY if v >= 65000 else SLATE_SECONDARY
    for v in df_top_energy["facility_total_energy_mwh"]
]

median_intensity = df_top_energy["mwh_per_yottaflop"].median()
intensity_colors = [
    TEAL_EFFICIENT if v <= median_intensity else CORAL_INTENSE
    for v in df_top_energy["mwh_per_yottaflop"]
]

fig2 = make_subplots(
    rows=1,
    cols=2,
    shared_yaxes=True,
    horizontal_spacing=0.06,
)

# Panel 1: Total Training Energy
fig2.add_trace(
    go.Bar(
        y=df_top_energy["model_label"],
        x=df_top_energy["facility_total_energy_mwh"],
        orientation="h",
        marker_color=energy_colors,
        text=df_top_energy["facility_total_energy_mwh"].apply(lambda v: f" {v:,.0f} MWh"),
        textposition="outside",
        textfont=dict(size=11, color="#334155"),
        cliponaxis=False,
        showlegend=False,
    ),
    row=1,
    col=1,
)

# Panel 2: Energy Intensity
fig2.add_trace(
    go.Bar(
        y=df_top_energy["model_label"],
        x=df_top_energy["mwh_per_yottaflop"],
        orientation="h",
        marker_color=intensity_colors,
        text=df_top_energy["mwh_per_yottaflop"].apply(lambda v: f" {v:,.0f}"),
        textposition="outside",
        textfont=dict(size=11, color="#334155"),
        cliponaxis=False,
        showlegend=False,
    ),
    row=1,
    col=2,
)

# Subtle dashed median line on Panel 2
fig2.add_vline(
    x=median_intensity,
    line_dash="dash",
    line_color="#94A3B8",
    line_width=1.2,
    row=1,
    col=2,
)

fig2.update_layout(
    title=dict(
        text=(
            "<b>Biggest Energy Consumers vs. Compute Efficiency (Top 12 Disclosed Models, 2022–2026)</b><br>"
            "<span style='font-size:12px; color:#64748B;'>"
            "Larger clusters consume more total power, yet use less energy per 10²⁴ operations (Teal ≤ median, Coral > median)"
            "</span>"
        ),
        x=0.01,
        xanchor="left",
        y=0.96,
        yanchor="top",
    ),
    font=dict(family="Segoe UI, Arial, sans-serif", size=12, color="#2C3E50"),
    plot_bgcolor="white",
    paper_bgcolor="white",
    height=660,
    margin=dict(t=105, b=65, l=220, r=70),
    annotations=[
        dict(
            text="1. Total Training Energy (MWh)",
            x=0.0,
            xanchor="left",
            y=1.07,
            xref="x domain",
            yref="y domain",
            showarrow=False,
            font=dict(size=12, color="#373838"),
        ),
        dict(
            text="2. Energy Intensity: MWh per 10²⁴ FLOP (Lower = Better)",
            x=0.0,
            xanchor="left",
            y=1.07,
            xref="x2 domain",
            yref="y2 domain",
            showarrow=False,
            font=dict(size=12, color="#373838"),
        ),
        dict(
            text=f"Median: {median_intensity:,.0f}",
            x=median_intensity,
            y=11.45,
            xref="x2",
            yref="y2",
            xanchor="left",
            xshift=6,
            showarrow=False,
            font=dict(size=10.5, color="#64748B"),
        ),
    ],
)

fig2.update_xaxes(
    title_text="Total Training Energy (MWh)",
    range=[0, df_top_energy["facility_total_energy_mwh"].max() * 1.28],
    gridcolor="#F1F5F9",
    zerolinecolor="#CBD5E1",
    row=1,
    col=1,
)
fig2.update_xaxes(
    title_text="MWh per 10²⁴ Operations",
    range=[0, df_top_energy["mwh_per_yottaflop"].max() * 1.18],
    gridcolor="#F1F5F9",
    zerolinecolor="#CBD5E1",
    row=1,
    col=2,
)
fig2.update_yaxes(showgrid=False, linecolor="#CBD5E1")

fig2.show()
fig2.write_image("../assets/charts/chart2_energy_efficiency.png", width=1200, height=660, scale=1)

### 4. Training Budget Breakdown (FinOps)

In [6]:
# Filter models with valid breakdown (Total Cost > Electricity Cost)
df_valid_cost = df_master.dropna(
    subset=["hardware_infra_cost_usd", "facility_electricity_cost_usd", "total_cost_usd"]
).copy()

df_valid_cost = df_valid_cost[df_valid_cost["total_cost_usd"] > 0].copy()
df_valid_cost["electricity_share_pct"] = (
    df_valid_cost["facility_electricity_cost_usd"] / df_valid_cost["total_cost_usd"]
) * 100.0

# Industry-wide benchmarks across all disclosed budgets
total_valid_count = len(df_valid_cost)
median_elec_share = df_valid_cost["electricity_share_pct"].median()

# Top 10 models by total training budget
df_cost = (
    df_valid_cost.sort_values("total_cost_usd", ascending=False)
    .head(10)
    .copy()
)

# Dollar-weighted electricity share across Top 10 frontier budgets
top10_elec_share = (
    df_cost["facility_electricity_cost_usd"].sum() / df_cost["total_cost_usd"].sum()
) * 100.0

# Calculate training days and daily burn rate (USD/day)
df_cost["training_days"] = (df_cost["training_time_hours"] / 24.0).round(1)
df_cost["daily_burn_usd"] = np.where(
    df_cost["training_days"] > 0,
    df_cost["total_cost_usd"] / df_cost["training_days"],
    np.nan,
)

# Identify top expensive models excluded due to missing electricity/training-hour breakdown
excluded_expensive = (
    df_master[
        df_master["total_cost_usd"].notna()
        & df_master["facility_electricity_cost_usd"].isna()
    ]
    .sort_values("total_cost_usd", ascending=False)
    .head(3)["model_name"]
    .tolist()
)
excluded_note = (
    ", ".join(excluded_expensive)
    if excluded_expensive
    else "GPT-4, Gemini Ultra, Grok 4"
)

NAVY = "#1E3A5F"
CORAL = "#D95F43"

fig3 = go.Figure()

# Bottom bar: Hardware & Cluster Infrastructure CapEx
fig3.add_trace(
    go.Bar(
        name="Hardware & Cluster Infrastructure (CapEx)",
        x=df_cost["model_name"],
        y=df_cost["hardware_infra_cost_usd"],
        marker=dict(color=NAVY, line=dict(color="white", width=0.8)),
        customdata=np.stack(
            [
                df_cost["organization"],
                df_cost["chip_family"],
                df_cost["training_days"],
                df_cost["daily_burn_usd"],
                100.0 - df_cost["electricity_share_pct"],
            ],
            axis=-1,
        ),
        hovertemplate=(
            "<b>%{x}</b> (%{customdata[0]})<br>"
            "Hardware: %{customdata[1]}<br>"
            "Hardware & Infra Cost: $%{y:,.0f} (%{customdata[4]:.1f}% of budget)<br>"
            "Training Duration: %{customdata[2]:.0f} days (~$%{customdata[3]:,.0f}/day)<extra></extra>"
        ),
    )
)

# Top bar: Pure Electricity Bill OpEx + Total Budget & Power % Label on top
top_labels = [
    f"<b>${tot / 1e6:.1f}M</b><br><span style='font-size:10px; color:#D95F43;'>({pct:.0f}% power)</span>"
    for tot, pct in zip(df_cost["total_cost_usd"], df_cost["electricity_share_pct"])
]

fig3.add_trace(
    go.Bar(
        name="Electricity Bill (Power OpEx)",
        x=df_cost["model_name"],
        y=df_cost["facility_electricity_cost_usd"],
        marker=dict(color=CORAL, line=dict(color="white", width=0.8)),
        text=top_labels,
        textposition="outside",
        cliponaxis=False,
        textfont=dict(size=11, color="#1E293B"),
        customdata=np.stack(
            [
                df_cost["electricity_share_pct"],
                df_cost["total_cost_usd"],
                df_cost["primary_country"],
            ],
            axis=-1,
        ),
        hovertemplate=(
            "<b>%{x}</b> (%{customdata[2]})<br>"
            "Electricity Bill: $%{y:,.0f} (%{customdata[0]:.1f}% of budget)<br>"
            "Total Training Budget: $%{customdata[1]:,.0f}<extra></extra>"
        ),
    )
)

fig3.update_layout(
    barmode="stack",
    title=dict(
        text=(
            f"<b>AI Training Cost Structure: Chips Dominate (~{100 - top10_elec_share:.0f}%), Power Is Secondary (~{top10_elec_share:.0f}%)</b><br>"
            "<span style='font-size:12px; color:#64748B;'>"
            f"Top 10 disclosed training budgets (across all {total_valid_count} disclosed models)"
            "</span>"
        ),
        x=0.01,
        xanchor="left",
        y=0.96,
        yanchor="top",
    ),
    font=dict(family="Segoe UI, Arial, sans-serif", size=12, color="#2C3E50"),
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.03,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255,255,255,0.9)",
    ),
    height=650,
    margin=dict(t=105, b=185, l=80, r=50),
    annotations=[
        dict(
            text=(
                f"<i>Note: Based on {total_valid_count} models disclosing both total budget and training hours "
                f"(excludes undisclosed high-cost runs such as {excluded_note}).<br>"
                f"Top labels show <b>Total Cost ($M)</b> and <b>Electricity Share (%)</b>. "
                f"Interactive tooltips include daily burn rate (USD/day).</i>"
            ),
            x=0.0,
            xanchor="left",
            y=-0.33,
            yanchor="top",
            xref="paper",
            yref="paper",
            showarrow=False,
            align="left",
            font=dict(size=10.5, color="#64748B"),
        )
    ],
)

fig3.update_xaxes(
    title_text="AI Model (Ranked by Total Disclosed Training Budget)",
    title_standoff=16,
    tickangle=-20,
    showgrid=False,
    linecolor="#CBD5E1",
)
fig3.update_yaxes(
    title_text="Total Training Cost (USD)",
    range=[0, df_cost["total_cost_usd"].max() * 1.22],
    gridcolor="#F1F5F9",
    zerolinecolor="#CBD5E1",
    tickprefix="$",
)

fig3.show()
fig3.write_image("../assets/charts/chart3_cost_structure.png", width=1200, height=650, scale=1)

### 5. Predictive Scaling Regression & Hidden Demand

In [7]:
# 1. Prepare disclosed models (strictly positive energy)
df_scatter = df_master[
    df_master["log10_flop"].notna()
    & (df_master["facility_total_energy_mwh"] > 0.05)
].copy()
df_scatter["training_days"] = (df_scatter["training_time_hours"] / 24.0).round(1)
df_scatter["log10_energy"] = np.log10(df_scatter["facility_total_energy_mwh"])

def group_hardware_gen(chip):
    if chip in ["NVIDIA H100", "NVIDIA H200", "NVIDIA B200"]:
        return "Frontier NVIDIA (H100 / H200 / B200)"
    elif chip == "NVIDIA A100":
        return "NVIDIA A100"
    elif "Google TPU" in str(chip):
        return "Google TPUs (v3 – v5p)"
    else:
        return "Legacy / Other Hardware"

df_scatter["hw_group"] = df_scatter["chip_family"].apply(group_hardware_gen)

# 2. Fit Log-Log Predictive Regression on Modern Scale Regime (log10_flop >= 21.0)
# Avoids distortion from tiny pre-2020 runs while plotting the full spectrum
df_fit = df_scatter[df_scatter["log10_flop"] >= 21.0].copy()
slope, intercept = np.polyfit(df_fit["log10_flop"], df_fit["log10_energy"], 1)
r_val = np.corrcoef(df_fit["log10_flop"], df_fit["log10_energy"])[0, 1]
r_squared = r_val ** 2

# Residual standard error for +/- 1 SD prediction band
y_pred_hist = slope * df_fit["log10_flop"] + intercept
resid_std = np.std(df_fit["log10_energy"] - y_pred_hist)

# 3. Top 4 Undisclosed Frontier Models (known FLOPs, NULL energy)
df_undisclosed = (
    df_master[df_master["facility_total_energy_mwh"].isna() & df_master["log10_flop"].notna()]
    .sort_values("log10_flop", ascending=False)
    .head(4)
    .copy()
    .reset_index(drop=True)
)
df_undisclosed["predicted_mwh"] = 10 ** (slope * df_undisclosed["log10_flop"] + intercept)
df_undisclosed["rank_num"] = [str(i + 1) for i in range(len(df_undisclosed))]

# Slight X-nudge if two undisclosed markers sit within 0.08 log10_flop so #2 and #3 never overlap
x_plot_undisclosed = df_undisclosed["log10_flop"].tolist()
for idx in range(1, len(x_plot_undisclosed)):
    if abs(x_plot_undisclosed[idx - 1] - x_plot_undisclosed[idx]) < 0.09:
        x_plot_undisclosed[idx] = x_plot_undisclosed[idx - 1] - 0.11

max_x = max(27.3, df_undisclosed["log10_flop"].max() + 0.35) if len(df_undisclosed) > 0 else 27.3
x_line = np.linspace(19.5, max_x, 120)
y_line_mwh = 10 ** (slope * x_line + intercept)
y_upper_mwh = 10 ** (slope * x_line + intercept + resid_std)
y_lower_mwh = 10 ** (slope * x_line + intercept - resid_std)

color_map = {
    "Frontier NVIDIA (H100 / H200 / B200)": "#2A7B88",
    "NVIDIA A100": "#1E3A5F",
    "Google TPUs (v3 – v5p)": "#E69F00",
    "Legacy / Other Hardware": "#94A3B8",
}

fig4 = go.Figure()

# Prediction confidence band (subtle shaded corridor)
fig4.add_trace(
    go.Scatter(
        x=np.concatenate([x_line, x_line[::-1]]),
        y=np.concatenate([y_upper_mwh, y_lower_mwh[::-1]]),
        fill="toself",
        fillcolor="rgba(30, 58, 95, 0.08)",
        line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip",
        showlegend=True,
        name="Typical Estimation Range (±1 SD)",
    )
)

# Disclosed models
for group_name, color in color_map.items():
    sub = df_scatter[df_scatter["hw_group"] == group_name]
    if len(sub) == 0:
        continue
    fig4.add_trace(
        go.Scatter(
            x=sub["log10_flop"],
            y=sub["facility_total_energy_mwh"],
            mode="markers",
            name=group_name,
            marker=dict(
                color=color,
                size=np.clip(np.sqrt(sub["pue_adjusted_co2_tons"].fillna(1)) * 0.13 + 6, 6, 28),
                opacity=0.52,
                line=dict(width=0.8, color="white"),
            ),
            customdata=np.stack(
                [
                    sub["model_name"],
                    sub["organization"],
                    sub["chip_family"],
                    sub["training_days"].fillna(0),
                    sub["pue_adjusted_co2_tons"].fillna(0),
                ],
                axis=-1,
            ),
            hovertemplate=(
                "<b>%{customdata[0]}</b> (%{customdata[1]})<br>"
                "Hardware: %{customdata[2]}<br>"
                "Computing Power: 10<sup>%{x:.1f}</sup> FLOP<br>"
                "Disclosed Energy: %{y:,.0f} MWh (%{customdata[3]:.0f} days)<br>"
                "Carbon Footprint: %{customdata[4]:,.0f} tons CO₂<extra></extra>"
            ),
        )
    )

# Scaling Law Regression Line
fig4.add_trace(
    go.Scatter(
        x=x_line,
        y=y_line_mwh,
        mode="lines",
        name=f"Modern Scaling Trendline (R² = {r_squared:.2f}, β = {slope:.2f})",
        line=dict(color="#1E3A5F", width=2.4, dash="dash"),
        hovertemplate=(
            "<b>Scaling Law Benchmark</b><br>"
            "Computing Power: 10<sup>%{x:.1f}</sup> FLOP<br>"
            "Expected Energy: %{y:,.0f} MWh<extra></extra>"
        ),
    )
)

# Highlighted Estimated Undisclosed Frontier Models (Numbered Coral Markers)
if len(df_undisclosed) > 0:
    fig4.add_trace(
        go.Scatter(
            x=x_plot_undisclosed,
            y=df_undisclosed["predicted_mwh"],
            mode="markers+text",
            name="Estimated Undisclosed Models (1–4)",
            marker=dict(
                symbol="circle",
                size=19,
                color="#D95F43",
                line=dict(width=2, color="white"),
            ),
            text=df_undisclosed["rank_num"],
            textposition="middle center",
            textfont=dict(size=10, color="white", family="Segoe UI, Arial, sans-serif"),
            customdata=np.stack(
                [
                    df_undisclosed["model_name"],
                    df_undisclosed["organization"],
                    df_undisclosed["publication_year"],
                    df_undisclosed["log10_flop"],
                ],
                axis=-1,
            ),
            hovertemplate=(
                "<b>#%{text} %{customdata[0]}</b> (%{customdata[1]}, %{customdata[2]})<br>"
                "<i>Undisclosed training hours — Model Estimate</i><br>"
                "Computing Power: 10<sup>%{customdata[3]:.2f}</sup> FLOP<br>"
                "Estimated Energy: <b>~%{y:,.0f} MWh</b><extra></extra>"
            ),
        )
    )

annotations_list = []

grok3 = df_scatter[df_scatter["model_name"].str.contains("Grok 3", case=False, na=False)].head(1)
if len(grok3) > 0:
    annotations_list.append(
        dict(
            x=grok3["log10_flop"].values[0],
            y=np.log10(grok3["facility_total_energy_mwh"].values[0]),
            text=f"<b>Benchmark: {grok3['model_name'].values[0]}</b><br>{grok3['facility_total_energy_mwh'].values[0]:,.0f} MWh (Disclosed)",
            showarrow=True,
            arrowhead=2,
            arrowcolor="#1E3A5F",
            ax=-120,
            ay=-18,
            bgcolor="rgba(255,255,255,0.94)",
            bordercolor="#CBD5E1",
            borderwidth=1,
            borderpad=6,
            font=dict(size=11, color="#1E3A5F"),
        )
    )

if len(df_undisclosed) > 0:
    lines = ["<b>Estimated Energy — Top Undisclosed Models</b>"]
    for _, r in df_undisclosed.iterrows():
        lines.append(
            f"<span style='color:#D95F43;'><b>#{r['rank_num']}</b></span> <b>{r['model_name']}</b> ({r['organization']}): <b>~{r['predicted_mwh']:,.0f} MWh</b>"
        )
    box_text = "<br>".join(lines)

    annotations_list.append(
        dict(
            text=box_text,
            x=0.985,
            xanchor="right",
            y=0.05,
            yanchor="bottom",
            xref="paper",
            yref="paper",
            showarrow=False,
            align="left",
            bgcolor="rgba(255, 250, 248, 0.96)",
            bordercolor="#D95F43",
            borderwidth=1.2,
            borderpad=10,
            font=dict(size=11, color="#1E293B"),
        )
    )

annotations_list.append(
    dict(
        text=(
            f"<i>Note: Scaling regression fitted on modern compute scale (≥10²¹ FLOP; slope β = {slope:.2f}, so 10× compute ≈ {10**slope:.1f}× energy). "
            "Bubble size reflects CO₂ emissions.<br>"
            "Models positioned <b>below</b> the dashed trendline achieved higher energy efficiency than the historical scaling average.</i>"
        ),
        x=0.0,
        xanchor="left",
        y=-0.18,
        yanchor="top",
        xref="paper",
        yref="paper",
        showarrow=False,
        align="left",
        font=dict(size=10.5, color="#64748B"),
    )
)

fig4.update_layout(
    title=dict(
        text=(
            f"<b>Estimating Hidden AI Energy Demand: Compute Explains {r_squared * 100:.0f}% of Energy Growth</b><br>"
            "<span style='font-size:12px; color:#64748B;'>"
            "Log-scale regression across modern disclosed runs projects electricity consumption for top undisclosed frontier releases"
            "</span>"
        ),
        x=0.01,
        xanchor="left",
        y=0.96,
        yanchor="top",
    ),
    font=dict(family="Segoe UI, Arial, sans-serif", size=12, color="#2C3E50"),
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255,255,255,0.9)",
    ),
    height=700,
    margin=dict(t=155, b=130, l=80, r=55),
    annotations=annotations_list,
)

fig4.update_xaxes(
    title_text="Training Computing Power (Log₁₀ FLOP — Each +1 = 10x More Compute)",
    title_standoff=14,
    range=[19.2, max_x + 0.35],
    gridcolor="#F1F5F9",
    linecolor="#CBD5E1",
)
fig4.update_yaxes(
    title_text="Total Energy Consumed (MWh — Log Scale)",
    type="log",
    range=[0.5, 6.7],
    tickvals=[10, 100, 1000, 10000, 100000, 1000000],
    ticktext=["10", "100", "1k", "10k", "100k", "1M"],
    gridcolor="#F1F5F9",
    linecolor="#CBD5E1",
)

fig4.show()
fig4.write_image("../assets/charts/chart4_predictive_scaling.png", width=1200, height=700, scale=1)

### 6. Geopolitical Grid Trade-Off & Site Selection

In [8]:
# 1. Clean country-level metrics directly from df_master (excluding unassigned/multinational)
exclude_countries = ["Unknown", "Multinational", "Other / Multinational", "Other", ""]

df_all_hubs = (
    df_master[
        df_master["primary_country"].notna()
        & (~df_master["primary_country"].isin(exclude_countries))
    ]
    .groupby("primary_country", as_index=False)
    .agg(
        hist_mwh=("facility_total_energy_mwh", "sum"),
        hist_co2=("pue_adjusted_co2_tons", "sum"),
        disclosed_runs=("facility_total_energy_mwh", "count"),
        tariff_usd_mwh=("tariff_usd_mwh", "median"),
        carbon_gco2_kwh=("carbon_gco2_kwh", "median"),
    )
)

# Panel 1 dataset: Countries with actual disclosed historical energy (> 0 MWh)
df_hist_panel = (
    df_all_hubs[df_all_hubs["hist_mwh"] > 0]
    .sort_values("hist_mwh", ascending=True)
    .copy()
)
total_disclosed_mwh = df_hist_panel["hist_mwh"].sum()
df_hist_panel["energy_share_pct"] = (df_hist_panel["hist_mwh"] / total_disclosed_mwh) * 100.0

# Panel 2 dataset: Candidate hubs with valid tariff & grid carbon intensity
df_sim = df_all_hubs.dropna(subset=["tariff_usd_mwh", "carbon_gco2_kwh"]).copy()
df_sim["sim_cost_usd_m"] = (50000.0 * df_sim["tariff_usd_mwh"]) / 1e6
df_sim["sim_co2_tons"] = (50000.0 * 1000.0 * df_sim["carbon_gco2_kwh"]) / 1e6

med_cost = df_sim["sim_cost_usd_m"].median()
med_co2 = df_sim["sim_co2_tons"].median()

fig5 = make_subplots(
    rows=1,
    cols=2,
    horizontal_spacing=0.12,
    column_widths=[0.44, 0.56],
)

# Panel 1: Minimalist Log-Scale Lollipop Chart (100% legible from 2 MWh to 786k MWh)
for _, r in df_hist_panel.iterrows():
    is_us = r["primary_country"] == "United States"
    fig5.add_shape(
        type="line",
        x0=1,
        x1=r["hist_mwh"],
        y0=r["primary_country"],
        y1=r["primary_country"],
        line=dict(color="#1E3A5F" if is_us else "#CBD5E1", width=2.5 if is_us else 1.8),
        row=1,
        col=1,
    )

point_colors_p1 = ["#1E3A5F" if c == "United States" else "#64748B" for c in df_hist_panel["primary_country"]]
point_sizes_p1 = [13 if c == "United States" else 10 for c in df_hist_panel["primary_country"]]
labels_p1 = [
    f"  {mwh:,.0f} MWh ({pct:.1f}%)" if pct >= 1.0 else f"  {mwh:,.0f} MWh (<1%)"
    for mwh, pct in zip(df_hist_panel["hist_mwh"], df_hist_panel["energy_share_pct"])
]

fig5.add_trace(
    go.Scatter(
        y=df_hist_panel["primary_country"],
        x=df_hist_panel["hist_mwh"],
        mode="markers+text",
        marker=dict(color=point_colors_p1, size=point_sizes_p1, line=dict(width=1.5, color="white")),
        text=labels_p1,
        textposition="middle right",
        textfont=dict(size=11, color="#1E293B"),
        cliponaxis=False,
        showlegend=False,
        hovertemplate="<b>%{y}</b><br>Disclosed Energy: %{x:,.0f} MWh<extra></extra>",
    ),
    row=1,
    col=1,
)

# Panel 2: Minimalist Scatter (Teal highlight for Canada, clean Slate for peers)
point_colors_p2 = ["#0F766E" if c == "Canada" else "#64748B" for c in df_sim["primary_country"]]
point_sizes_p2 = [15 if c == "Canada" else 11 for c in df_sim["primary_country"]]

fig5.add_trace(
    go.Scatter(
        x=df_sim["sim_cost_usd_m"],
        y=df_sim["sim_co2_tons"],
        mode="markers",
        marker=dict(size=point_sizes_p2, color=point_colors_p2, line=dict(width=1.5, color="white")),
        showlegend=False,
        customdata=np.stack([df_sim["primary_country"], df_sim["tariff_usd_mwh"], df_sim["carbon_gco2_kwh"]], axis=-1),
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Simulated Bill: $%{x:.2f}M ($%{customdata[1]:.0f}/MWh)<br>"
            "Simulated Carbon: %{y:,.0f} tons CO₂ (%{customdata[2]:.0f} gCO₂/kWh)<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)

# Subtle median reference lines
fig5.add_vline(x=med_cost, line_dash="dot", line_color="#CBD5E1", line_width=1.2, row=1, col=2)
fig5.add_hline(y=med_co2, line_dash="dot", line_color="#CBD5E1", line_width=1.2, row=1, col=2)

x_min_p2 = max(0, df_sim["sim_cost_usd_m"].min() * 0.72)
x_max_p2 = df_sim["sim_cost_usd_m"].max() * 1.18
y_max_p2 = df_sim["sim_co2_tons"].max() * 1.18

country_annotations = []
for _, r in df_sim.iterrows():
    c_name = r["primary_country"]
    is_optimal = c_name == "Canada"
    y_offset = -24 if c_name == "United Kingdom" else 22
    country_annotations.append(
        dict(
            x=r["sim_cost_usd_m"],
            y=r["sim_co2_tons"],
            xref="x2",
            yref="y2",
            text=(
                f"<b>{c_name}</b><br>"
                f"<span style='font-size:10px;color:{'#0F766E' if is_optimal else '#64748B'};'>"
                f"${r['sim_cost_usd_m']:.1f}M · {r['sim_co2_tons']/1000:.1f}k t"
                f"</span>"
            ),
            showarrow=False,
            yshift=y_offset,
            font=dict(size=11, color="#0F766E" if is_optimal else "#1E293B"),
        )
    )

us_share = df_hist_panel.loc[df_hist_panel["primary_country"] == "United States", "energy_share_pct"].values[0]

fig5.update_layout(
    title=dict(
        text=(
            "<b>Where Is the Best Hub to Train Frontier AI?</b><br>"
            "<span style='font-size:12px; color:#64748B;'>"
            f"The US concentrates {us_share:.0f}% of disclosed training energy, while Canada offers the lowest cost and cleanest grid"
            "</span>"
        ),
        x=0.01,
        xanchor="left",
        y=0.96,
        yanchor="top",
    ),
    font=dict(family="Segoe UI, Arial, sans-serif", size=12, color="#2C3E50"),
    plot_bgcolor="white",
    paper_bgcolor="white",
    showlegend=False,
    height=620,
    margin=dict(t=105, b=65, l=125, r=45),
    annotations=[
        dict(
            text="1. Historical Disclosed Energy (Log Scale)",
            x=0.0,
            xanchor="left",
            y=1.07,
            xref="x domain",
            yref="y domain",
            showarrow=False,
            font=dict(size=12, color="#1E293B"),
        ),
        dict(
            text="2. 50,000 MWh Cluster Simulation: Cost vs. Carbon",
            x=0.0,
            xanchor="left",
            y=1.07,
            xref="x2 domain",
            yref="y2 domain",
            showarrow=False,
            font=dict(size=12, color="#1E293B"),
        ),
        dict(
            text="Optimal Hub (Low Cost & Clean Grid)",
            x=x_min_p2 + 0.25,
            y=med_co2 * 0.92,
            xref="x2",
            yref="y2",
            xanchor="left",
            yanchor="top",
            showarrow=False,
            font=dict(size=10, color="#0F766E"),
        ),
    ] + country_annotations,
)

fig5.update_xaxes(
    title_text="Total Disclosed Training Energy (MWh — Log Scale)",
    title_standoff=14,
    type="log",
    range=[0, 7.6],
    tickvals=[1, 10, 100, 1000, 10000, 100000, 1000000],
    ticktext=["1", "10", "100", "1k", "10k", "100k", "1M"],
    gridcolor="#F8FAFC",
    linecolor="#CBD5E1",
    row=1,
    col=1,
)
fig5.update_yaxes(showgrid=False, linecolor="#CBD5E1", row=1, col=1)

fig5.update_xaxes(
    title_text="Simulated Electricity Bill ($ Millions USD)",
    title_standoff=14,
    range=[x_min_p2, x_max_p2],
    tickprefix="$",
    ticksuffix="M",
    gridcolor="#F8FAFC",
    zerolinecolor="#CBD5E1",
    row=1,
    col=2,
)
fig5.update_yaxes(
    title_text="Simulated Emissions (Tons CO₂)",
    range=[0, y_max_p2],
    gridcolor="#F8FAFC",
    zerolinecolor="#CBD5E1",
    row=1,
    col=2,
)

fig5.show()
fig5.write_image("../assets/charts/chart5_geopolitical_grid.png", width=1200, height=620, scale=1)

### 7. AI Governance & Reporting Risk

In [9]:
# Prepare AI Governance & Accessibility segmentation safely from df_master
df_gov = df_master[df_master["publication_year"] >= 2018].copy()

access_col = None
for candidate in ["model_accessibility", "accessibility", "Model accessibility"]:
    if candidate in df_gov.columns:
        access_col = candidate
        break

def classify_governance(row):
    if access_col and pd.notna(row[access_col]):
        val = str(row[access_col]).lower()
        if "open" in val:
            return "Open-Weights / Public"
        elif "api" in val or "hosted" in val or "closed" in val or "unreleased" in val:
            return "Closed / Commercial API"
    name_org = f"{row.get('model_name', '')} {row.get('organization', '')}".lower()
    open_keywords = [
        "llama", "mistral", "mixtral", "deepseek", "qwen", "gemma",
        "falcon", "olmo", "pythia", "bloom", "opt-", "t5", "bert", "yi-", "phi-"
    ]
    if any(k in name_org for k in open_keywords):
        return "Open-Weights / Public"
    return "Closed / Commercial API"

df_gov["release_model"] = df_gov.apply(classify_governance, axis=1)
df_gov["has_energy"] = df_gov["facility_total_energy_mwh"].notna()

# Left Panel Data: Annual Share (%) of Open-Weights vs Closed Models (2018-2026)
df_gov_year = (
    df_gov.groupby(["publication_year", "release_model"], as_index=False)
    .agg(models_count=("model_id", "count"))
)
df_year_tot = df_gov.groupby("publication_year", as_index=False).agg(year_total=("model_id", "count"))
df_gov_year = df_gov_year.merge(df_year_tot, on="publication_year")
df_gov_year["share_pct"] = (df_gov_year["models_count"] / df_gov_year["year_total"]) * 100.0
df_gov_year["year_label"] = df_gov_year["publication_year"].apply(
    lambda y: f"{int(y)}*" if int(y) == 2026 else str(int(y))
)

# Right Panel Data: Energy Disclosure Rate (%) ordered consistently (Open-Weights first, Closed second)
category_order = ["Open-Weights / Public", "Closed / Commercial API"]
df_gov_summary = (
    df_gov.groupby("release_model", as_index=False)
    .agg(
        total_models=("model_id", "count"),
        disclosed_models=("has_energy", "sum"),
        median_log10_flop=("log10_flop", "median"),
    )
)
df_gov_summary["disclosure_pct"] = (
    df_gov_summary["disclosed_models"] / df_gov_summary["total_models"]
) * 100.0
df_gov_summary["release_model"] = pd.Categorical(
    df_gov_summary["release_model"], categories=category_order, ordered=True
)
df_gov_summary = df_gov_summary.sort_values("release_model")

overall_disclosure_pct = (df_gov["has_energy"].sum() / len(df_gov)) * 100.0

NAVY = "#1E3A5F"
TEAL = "#2A7B88"

fig6 = make_subplots(
    rows=1,
    cols=2,
    horizontal_spacing=0.14,
    column_widths=[0.58, 0.42],
)

# Left Panel: 100% Stacked Bars over time (consistent color & order)
for rel_type, color in [
    ("Open-Weights / Public", TEAL),
    ("Closed / Commercial API", NAVY),
]:
    sub = df_gov_year[df_gov_year["release_model"] == rel_type].sort_values("publication_year")
    fig6.add_trace(
        go.Bar(
            x=sub["year_label"],
            y=sub["share_pct"],
            name=rel_type,
            marker=dict(color=color, line=dict(color="white", width=0.6)),
            text=sub["share_pct"].apply(lambda p: f"{p:.0f}%" if p >= 12 else ""),
            textposition="inside",
            insidetextanchor="middle",
            textfont=dict(size=10.5, color="white"),
            customdata=np.stack([sub["models_count"], sub["year_total"]], axis=-1),
            hovertemplate=(
                "<b>%{x} — " + rel_type + "</b><br>"
                "Share: %{y:.1f}% (%{customdata[0]} of %{customdata[1]} models)<extra></extra>"
            ),
        ),
        row=1,
        col=1,
    )

# Right Panel: Disclosure Rate (%) with matching category colors (Teal vs Navy)
bar_colors_r = [TEAL, NAVY]
fig6.add_trace(
    go.Bar(
        x=df_gov_summary["release_model"],
        y=df_gov_summary["disclosure_pct"],
        showlegend=False,
        marker=dict(color=bar_colors_r, line=dict(width=0)),
        width=0.45,
        text=df_gov_summary.apply(
            lambda r: f"<b>{r['disclosure_pct']:.1f}%</b><br><span style='font-size:10.5px;color:#64748B;'>({int(r['disclosed_models'])}/{int(r['total_models'])})</span>",
            axis=1,
        ),
        textposition="outside",
        cliponaxis=False,
        textfont=dict(size=12, color="#1E293B"),
        customdata=df_gov_summary["median_log10_flop"],
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Energy Disclosure Rate: %{y:.1f}%<br>"
            "Median Computing Power: 10<sup>%{customdata:.1f}</sup> FLOP<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)

# Subtle industry-average benchmark line on Panel 2
fig6.add_hline(
    y=overall_disclosure_pct,
    line_dash="dot",
    line_color="#94A3B8",
    line_width=1.3,
    row=1,
    col=2,
)

fig6.update_layout(
    barmode="stack",
    title=dict(
        text=(
            "<b>Commercial Closure Widens the Energy Transparency Gap</b><br>"
            "<span style='font-size:12px; color:#64748B;'>"
            "Open-weight share fell from 56% (2020) to 35% (2026*), while closed API releases disclose energy data nearly half as often"
            "</span>"
        ),
        x=0.01,
        xanchor="left",
        y=0.96,
        yanchor="top",
    ),
    font=dict(family="Segoe UI, Arial, sans-serif", size=12, color="#2C3E50"),
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.06,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255,255,255,0.9)",
    ),
    height=620,
    margin=dict(t=125, b=85, l=65, r=45),
    annotations=[
        dict(
            text="1. Annual Share of Model Releases (2018–2026*)",
            x=0.0,
            xanchor="left",
            y=1.05,
            xref="x domain",
            yref="y domain",
            showarrow=False,
            font=dict(size=12, color="#1E293B"),
        ),
        dict(
            text="2. Energy Data Disclosure Rate (%)",
            x=0.0,
            xanchor="left",
            y=1.05,
            xref="x2 domain",
            yref="y2 domain",
            showarrow=False,
            font=dict(size=12, color="#1E293B"),
        ),
        dict(
            text=f"Industry Avg: {overall_disclosure_pct:.1f}%",
            x=1.42,
            y=overall_disclosure_pct,
            xref="x2",
            yref="y2",
            xanchor="right",
            yanchor="bottom",
            yshift=3,
            showarrow=False,
            font=dict(size=10, color="#64748B"),
        ),
        dict(
            text="<i>Note: Notable AI models released 2018–2026 (*through November). Right panel labels show disclosed vs. total models per strategy.</i>",
            x=0.0,
            xanchor="left",
            y=-0.14,
            yanchor="top",
            xref="paper",
            yref="paper",
            showarrow=False,
            font=dict(size=10.5, color="#64748B"),
        ),
    ],
)

fig6.update_xaxes(title_text="Release Year (*2026 YTD)", showgrid=False, linecolor="#CBD5E1", row=1, col=1)
fig6.update_yaxes(title_text="Share of Annual Releases (%)", range=[0, 102], gridcolor="#F8FAFC", row=1, col=1)

fig6.update_xaxes(title_text="Release Strategy", showgrid=False, linecolor="#CBD5E1", row=1, col=2)
fig6.update_yaxes(
    title_text="Models Disclosing Energy (%)",
    range=[0, df_gov_summary["disclosure_pct"].max() * 1.32],
    gridcolor="#F8FAFC",
    zerolinecolor="#CBD5E1",
    row=1,
    col=2,
)

fig6.show()
fig6.write_image("../assets/charts/chart6_governance_risk.png", width=1200, height=620, scale=1)

### 8. Competitive Landscape (Top 10 AI Labs)

In [10]:
# Top 10 AI Organizations Leaderboard: Entity resolution + Output, Peak Scale & Energy Transparency
def normalize_org(org_str):
    lead = str(org_str).split(",")[0].strip()
    if lead in ["Google", "DeepMind", "Google DeepMind", "Google Brain"]:
        return "Google / DeepMind"
    if lead in ["Meta AI", "Facebook AI Research", "Facebook", "FAIR"]:
        return "Meta / FAIR"
    return lead

df_orgs = (
    df_master.dropna(subset=["organization"])
    .assign(org_clean=lambda d: d["organization"].apply(normalize_org))
    .query("org_clean != '' and org_clean != 'None' and org_clean != 'nan'")
    .groupby("org_clean", as_index=False)
    .agg(
        total_models=("model_id", "count"),
        disclosed_models=("facility_total_energy_mwh", "count"),
        max_log10_flop=("log10_flop", "max"),
        total_disclosed_mwh=("facility_total_energy_mwh", "sum"),
    )
    .sort_values(["total_models", "max_log10_flop"], ascending=[False, False])
    .head(10)
    .sort_values("total_models", ascending=True)
    .copy()
)

df_orgs["transparency_pct"] = (
    (df_orgs["disclosed_models"] / df_orgs["total_models"]) * 100.0
).round(1)

top10_share_pct = (df_orgs["total_models"].sum() / len(df_master)) * 100.0
benchmark_pct = (df_orgs["disclosed_models"].sum() / df_orgs["total_models"].sum()) * 100.0

NAVY_LEADER = "#1E3A5F"
SLATE_PEER = "#7A99B8"
TEAL_GOOD = "#165303"
CORAL_LOW = "#8EAD88"

vol_colors = [
    NAVY_LEADER if c in ["Google / DeepMind", "OpenAI", "Meta / FAIR"] else SLATE_PEER
    for c in df_orgs["org_clean"]
]
transp_colors = [
    TEAL_GOOD if p >= benchmark_pct else CORAL_LOW
    for p in df_orgs["transparency_pct"]
]

fig7 = make_subplots(
    rows=1,
    cols=2,
    shared_yaxes=True,
    horizontal_spacing=0.07,
    column_widths=[0.54, 0.46],
)

# Left Panel: Total Models Released + Peak Compute Scale
fig7.add_trace(
    go.Bar(
        y=df_orgs["org_clean"],
        x=df_orgs["total_models"],
        orientation="h",
        marker_color=vol_colors,
        text=df_orgs.apply(
            lambda r: f" <b>{int(r['total_models'])}</b> <span style='color:#64748B;'>(Peak: 10<sup>{r['max_log10_flop']:.1f}</sup>)</span>"
            if pd.notna(r["max_log10_flop"])
            else f" <b>{int(r['total_models'])}</b>",
            axis=1,
        ),
        textposition="outside",
        cliponaxis=False,
        textfont=dict(size=11, color="#1E293B"),
        customdata=df_orgs["total_disclosed_mwh"],
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Total Notable Models: %{x}<br>"
            "Disclosed Training Energy: %{customdata:,.0f} MWh<extra></extra>"
        ),
    ),
    row=1,
    col=1,
)

# Right Panel: Transparency Rate (%)
fig7.add_trace(
    go.Bar(
        y=df_orgs["org_clean"],
        x=df_orgs["transparency_pct"],
        orientation="h",
        marker_color=transp_colors,
        text=df_orgs.apply(
            lambda r: f" <b>{r['transparency_pct']:.0f}%</b> <span style='color:#64748B;'>({int(r['disclosed_models'])}/{int(r['total_models'])})</span>",
            axis=1,
        ),
        textposition="outside",
        cliponaxis=False,
        textfont=dict(size=11, color="#1E293B"),
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Energy Transparency: %{x:.1f}%<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)

# Top-10 aggregate disclosure benchmark line on Panel 2
fig7.add_vline(
    x=benchmark_pct,
    line_dash="dot",
    line_color="#94A3B8",
    line_width=1.3,
    row=1,
    col=2,
)

fig7.update_layout(
    title=dict(
        text=(
            f"<b>Top 10 AI Labs Drive {top10_share_pct:.0f}% of All Notable Models</b><br>"
            "<span style='font-size:12px; color:#64748B;'>"
            f"Model output & peak compute (log₁₀ FLOP) vs. energy disclosure rate (Teal ≥{benchmark_pct:.0f}% Top-10 avg, Coral below avg)"
            "</span>"
        ),
        x=0.01,
        xanchor="left",
        y=0.96,
        yanchor="top",
    ),
    font=dict(family="Segoe UI, Arial, sans-serif", size=12, color="#2C3E50"),
    plot_bgcolor="white",
    paper_bgcolor="white",
    showlegend=False,
    height=620,
    margin=dict(t=105, b=65, l=165, r=80),
    annotations=[
        dict(
            text="1. Model Output & Peak Compute Scale",
            x=0.0,
            xanchor="left",
            y=1.08,
            xref="x domain",
            yref="y domain",
            showarrow=False,
            font=dict(size=12, color="#1E293B"),
        ),
        dict(
            text="2. Energy Data Disclosure Rate (%)",
            x=0.0,
            xanchor="left",
            y=1.08,
            xref="x2 domain",
            yref="y2 domain",
            showarrow=False,
            font=dict(size=12, color="#1E293B"),
        ),
        dict(
            text=f"Top-10 Avg: {benchmark_pct:.0f}%",
            x=benchmark_pct,
            y=-0.52,
            xref="x2",
            yref="y2",
            xanchor="center",
            showarrow=False,
            font=dict(size=10, color="#64748B"),
            bgcolor="rgba(255,255,255,0.92)",
        ),
    ],
)

fig7.update_xaxes(
    title_text="Notable Models Released",
    title_standoff=14,
    range=[0, df_orgs["total_models"].max() * 1.42],
    gridcolor="#F8FAFC",
    zerolinecolor="#CBD5E1",
    row=1,
    col=1,
)
fig7.update_xaxes(
    title_text="Models Disclosing Energy (%)",
    title_standoff=14,
    range=[0, max(df_orgs["transparency_pct"].max() * 1.32, 55)],
    ticksuffix="%",
    gridcolor="#F8FAFC",
    zerolinecolor="#CBD5E1",
    row=1,
    col=2,
)
fig7.update_yaxes(showgrid=False, linecolor="#CBD5E1")

fig7.show()
fig7.write_image("../assets/charts/chart7_competitive_landscape.png", width=1200, height=620, scale=1)

### 9. Physics-Informed Energy Imputation & Scaling Uncertainty

In [23]:
import numpy as np
import plotly.graph_objects as go
import statsmodels.api as sm

# 1. Prepare verified dataset & fit scaling trendline
valid_imp = (
    df_master.dropna(subset=["training_compute_flop", "facility_total_energy_mwh"])
    .query("training_compute_flop > 0 and facility_total_energy_mwh > 0.05")
    .assign(
        log_flop=lambda d: np.log10(d["training_compute_flop"]),
        log_energy=lambda d: np.log10(d["facility_total_energy_mwh"]),
    )
)

ols_model = sm.OLS(valid_imp["log_energy"], sm.add_constant(valid_imp["log_flop"])).fit()
beta_slope = ols_model.params["log_flop"]

def predict_mwh_bounds(log_flop_series, alpha=0.05):
    X_design = sm.add_constant(log_flop_series, has_constant="add")
    summary = ols_model.get_prediction(X_design).summary_frame(alpha=alpha)
    return 10 ** summary["mean"], 10 ** summary["obs_ci_lower"], 10 ** summary["obs_ci_upper"]

# 2. Estimate energy for undisclosed frontier models
missing_energy = (
    df_master[df_master["facility_total_energy_mwh"].isna() & (df_master["training_compute_flop"] > 0)]
    .assign(log_flop=lambda d: np.log10(d["training_compute_flop"]))
)

p50, low95, high95 = predict_mwh_bounds(missing_energy["log_flop"], alpha=0.05)
_, low80, high80 = predict_mwh_bounds(missing_energy["log_flop"], alpha=0.20)

missing_energy = missing_energy.assign(
    imputed_energy_mwh_p50=p50,
    imputed_energy_mwh_low80=low80,
    imputed_energy_mwh_high80=high80,
    imputed_energy_mwh_low95=low95,
    imputed_energy_mwh_high95=high95,
)

# 3. Build regression grid across full observed + estimated domain
x_min = valid_imp["log_flop"].min()
x_max = max(valid_imp["log_flop"].max(), missing_energy["log_flop"].max()) + 0.25
x_grid = np.linspace(x_min, x_max, 220)

grid_p50, grid_low95, grid_high95 = predict_mwh_bounds(x_grid, alpha=0.05)
_, grid_low80, grid_high80 = predict_mwh_bounds(x_grid, alpha=0.20)

# Top 4 undisclosed models (sorted by energy descending) with slight visual separation if points overlap
top_imputed = missing_energy.nlargest(4, "imputed_energy_mwh_p50").reset_index(drop=True)
top_imputed["rank_num"] = [str(i + 1) for i in range(len(top_imputed))]

x_imp_plot = top_imputed["log_flop"].tolist()
for idx in range(1, len(x_imp_plot)):
    if abs(x_imp_plot[idx - 1] - x_imp_plot[idx]) < 0.09:
        x_imp_plot[idx] = x_imp_plot[idx - 1] - 0.11

# Key verified benchmarks (sorted top-to-bottom by energy for clean vertical spacing)
verified_targets = ["Grok 3", "Gemini 1.0 Ultra", "Llama 3.1-405B", "GPT-4 (Mar 2023)"]
top_verified = (
    valid_imp[valid_imp["model_name"].isin(verified_targets)]
    .sort_values("facility_total_energy_mwh", ascending=False)
    .reset_index(drop=True)
)

# 4. Construct Minimalist Figure
NAVY = "#1E3A5F"
TEAL = "#2A7B88"
TEAL_DARK = "#0F766E"
CORAL = "#D95F43"

fig8 = go.Figure()

def add_interval_band(x, lower, upper, fillcolor, label):
    fig8.add_trace(go.Scatter(
        x=np.concatenate([x, x[::-1]]),
        y=np.concatenate([upper, lower[::-1]]),
        fill="toself",
        fillcolor=fillcolor,
        line=dict(width=0),
        name=label,
        hoverinfo="skip",
    ))

add_interval_band(x_grid, grid_low95, grid_high95, "rgba(30, 58, 95, 0.05)", "Outer Range (95% of models)")
add_interval_band(x_grid, grid_low80, grid_high80, "rgba(42, 123, 136, 0.14)", "Typical Range (80% of models)")

# Expected Trendline
fig8.add_trace(go.Scatter(
    x=x_grid,
    y=grid_p50,
    mode="lines",
    line=dict(color=NAVY, width=2.4),
    name="Expected Energy Trend",
))

# Disclosed Models (subtle background)
fig8.add_trace(go.Scatter(
    x=valid_imp["log_flop"],
    y=valid_imp["facility_total_energy_mwh"],
    mode="markers",
    text=valid_imp["model_name"],
    marker=dict(size=6.5, color=TEAL, opacity=0.50, line=dict(width=0.5, color="white")),
    name=f"Disclosed Models ({len(valid_imp)})",
    hovertemplate="<b>%{text}</b><br>Disclosed Energy: %{y:,.0f} MWh<extra></extra>",
))

# Highlighted Disclosed Benchmarks
fig8.add_trace(go.Scatter(
    x=top_verified["log_flop"],
    y=top_verified["facility_total_energy_mwh"],
    mode="markers",
    text=top_verified["model_name"],
    marker=dict(size=10, color=TEAL_DARK, line=dict(width=1.8, color="white")),
    showlegend=False,
    hovertemplate="<b>%{text}</b> (Disclosed)<br>Energy: %{y:,.0f} MWh<extra></extra>",
))

# Highlighted Top 4 Estimated Models as crisp numbered circles (1–4)
fig8.add_trace(go.Scatter(
    x=x_imp_plot,
    y=top_imputed["imputed_energy_mwh_p50"],
    mode="markers+text",
    text=top_imputed["rank_num"],
    textposition="middle center",
    textfont=dict(size=10, color="white", family="Segoe UI, Arial, sans-serif"),
    customdata=np.column_stack([top_imputed["model_name"], top_imputed["imputed_energy_mwh_low80"], top_imputed["imputed_energy_mwh_high80"]]),
    marker=dict(size=19, color=CORAL, line=dict(width=2, color="white")),
    name="Estimated Closed Models (1–4)",
    hovertemplate=(
        "<b>#%{text} %{customdata[0]}</b> (Estimated)<br>"
        "Most Likely Energy: <b>%{y:,.0f} MWh</b><br>"
        "Typical Range: %{customdata[1]:,.0f} – %{customdata[2]:,.0f} MWh<extra></extra>"
    ),
))

# Real-world household benchmarks
for mwh_val in [10000, 100000]:
    fig8.add_hline(y=mwh_val, line_dash="dot", line_color="#CBD5E1", line_width=1)

# 5. Clean, Staggered Left-Side Callouts for Disclosed Models (Zero Overlap)
verified_layout = {
    "Grok 3": (-95, -24),
    "Gemini 1.0 Ultra": (-125, -36),
    "Llama 3.1-405B": (-145, -14),
    "GPT-4 (Mar 2023)": (-135, 16),
}

callouts = []
for _, r in top_verified.iterrows():
    name = r["model_name"]
    short_name = name.replace(" (Mar 2023)", "")
    ax_s, ay_s = verified_layout.get(name, (-100, -20))
    callouts.append(dict(
        x=r["log_flop"],
        y=np.log10(r["facility_total_energy_mwh"]),
        text=f"<b>{short_name}</b> · {r['facility_total_energy_mwh'] / 1000:.0f}k MWh",
        showarrow=True,
        arrowhead=0,
        arrowcolor="#94A3B8",
        arrowwidth=1,
        ax=ax_s,
        ay=ay_s,
        font=dict(size=10.5, color="#1E293B"),
    ))

# Minimalist Legend Card in Bottom-Right for Numbered Closed Models (1–4)
if len(top_imputed) > 0:
    lines = ["<span style='color:#64748B; font-size:10px;'><b>ESTIMATED CLOSED RELEASES</b></span>"]
    for _, r in top_imputed.iterrows():
        lines.append(
            f"<span style='color:{CORAL};'><b>#{r['rank_num']}</b></span> "
            f"<b>{r['model_name']}</b> · ~{r['imputed_energy_mwh_p50'] / 1000:.0f}k MWh"
        )
    callouts.append(dict(
        text="<br>".join(lines),
        x=0.985,
        y=0.05,
        xref="paper",
        yref="paper",
        xanchor="right",
        yanchor="bottom",
        showarrow=False,
        align="left",
        bgcolor="rgba(255, 255, 255, 0.94)",
        bordercolor="#E2E8F0",
        borderwidth=1,
        borderpad=9,
        font=dict(size=11, color="#1E293B"),
    ))

# Household reference labels on left edge
callouts.extend([
    dict(
        x=x_min - 0.15,
        y=np.log10(10000),
        text="≈ 1,000 homes / year",
        xanchor="left",
        yanchor="bottom",
        yshift=2,
        showarrow=False,
        font=dict(size=9.5, color="#94A3B8"),
    ),
    dict(
        x=x_min - 0.15,
        y=np.log10(100000),
        text="≈ 10,000 homes / year",
        xanchor="left",
        yanchor="bottom",
        yshift=2,
        showarrow=False,
        font=dict(size=9.5, color="#94A3B8"),
    ),
])

fig8.update_layout(
    title=dict(
        text=(
            "<b>How Much Electricity Does It Take to Train Frontier AI?</b><br>"
            "<span style='font-size:12px; color:#64748B;'>"
            f"Every 10× jump in computing power multiplies energy use by ~{10**beta_slope:.1f}× — allowing us to estimate closed releases"
            "</span>"
        ),
        x=0.01,
        xanchor="left",
        y=0.96,
        yanchor="top",
    ),
    font=dict(family="Segoe UI, Arial, sans-serif", size=12, color="#2C3E50"),
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.01, bgcolor="rgba(255,255,255,0.9)"),
    height=640,
    margin=dict(t=120, b=65, l=80, r=55),
    annotations=callouts,
)

fig8.update_xaxes(
    title_text="Training Computing Power (Each step = 100× more compute)",
    range=[x_min - 0.3, x_max + 0.45],
    tickvals=[18, 20, 22, 24, 26],
    ticktext=["10¹⁸", "10²⁰", "10²²", "10²⁴", "10²⁶"],
    gridcolor="#F1F5F9",
    linecolor="#CBD5E1",
)
fig8.update_yaxes(
    title_text="Total Training Electricity (MWh)",
    type="log",
    range=[-1.5, 6.8],
    tickvals=[0.1, 1, 10, 100, 1000, 10000, 100000, 1000000],
    ticktext=["0.1", "1", "10", "100", "1k", "10k", "100k", "1M"],
    gridcolor="#F1F5F9",
    linecolor="#CBD5E1",
)

fig8.show()
fig8.write_image("../assets/charts/chart8_imputation_uncertainty.png", width=1200, height=640, scale=2)

### 10. AI Economics Pareto Frontier & 12-Month Inference TCO

In [24]:
# Chart 9: AI Economics Pareto Frontier — High-Clarity Minimalist Print Version
pareto_df = df_master.dropna(subset=["Composite_Capability_Index", "total_cost_usd"]).copy()
pareto_df = pareto_df[
    (pareto_df["Composite_Capability_Index"] > 0) & (pareto_df["total_cost_usd"] > 0)
].copy()

# Consolidate duplicate minor revisions (e.g., DeepSeek-V3 vs DeepSeek-V3 (Mar 2025))
pareto_df["base_model"] = pareto_df["model_name"].str.replace(
    r"\s*\((Mar|May|Jun)\s*202[3-6]\)", "", regex=True
)
pareto_df = (
    pareto_df.sort_values(["Composite_Capability_Index", "total_cost_usd"], ascending=[False, True])
    .drop_duplicates(subset=["base_model"], keep="first")
    .sort_values("total_cost_usd")
    .reset_index(drop=True)
)

# Compute Pareto Frontier (Minimize Cost, Maximize Composite Capability Index)
frontier_rows = []
max_cap = -1.0
for _, row in pareto_df.iterrows():
    if row["Composite_Capability_Index"] > max_cap:
        frontier_rows.append(row)
        max_cap = row["Composite_Capability_Index"]

frontier = pd.DataFrame(frontier_rows).reset_index(drop=True)
non_frontier = pareto_df[~pareto_df["model_id"].isin(frontier["model_id"])].copy()

def fmt_usd_short(v):
    if v >= 1e6:
        return f"${v / 1e6:.1f}M"
    return f"${v / 1e3:.0f}k"

NAVY = "#1E3A5F"
CORAL = "#D95F43"
SLATE = "#94A3B8"

fig9 = go.Figure()

# Non-frontier models (clean muted circles)
fig9.add_trace(
    go.Scatter(
        x=non_frontier["total_cost_usd"],
        y=non_frontier["Composite_Capability_Index"],
        mode="markers",
        name="Other Evaluated Models",
        text=non_frontier["base_model"],
        customdata=non_frontier["organization"],
        marker=dict(size=10, color=SLATE, opacity=0.72, line=dict(width=1.2, color="white")),
        hovertemplate="<b>%{text}</b> (%{customdata})<br>Training Cost: $%{x:,.0f}<br>Capability Index: %{y:.1f}<extra></extra>",
    )
)

# Pareto-optimal frontier curve
fig9.add_trace(
    go.Scatter(
        x=frontier["total_cost_usd"],
        y=frontier["Composite_Capability_Index"],
        mode="lines+markers",
        name="Pareto Efficiency Frontier (Best Capability per $)",
        text=frontier["base_model"],
        line=dict(color=CORAL, width=2.4, dash="dash"),
        marker=dict(size=12, color=CORAL, symbol="diamond", line=dict(width=1.8, color="white")),
        customdata=frontier["organization"],
        hovertemplate="<b>%{text}</b> (%{customdata}) [Pareto Optimal]<br>Training Cost: $%{x:,.0f}<br>Capability Index: %{y:.1f}<extra></extra>",
    )
)

# 1. Collision-free annotations for ALL Pareto Frontier leaders (angled cleanly above/left)
frontier_custom_offsets = {
    "Llama 2-7B": (15, -24),
    "Falcon-40B": (-48, -26),
    "LLaMA-65B": (-52, -26),
    "Llama 2-70B": (-55, -26),
    "DeepSeek-R1": (-62, -28),
    "Llama 3.1-405B": (-72, -24),
}

annotations_9 = []
for _, r in frontier.iterrows():
    name = r["base_model"]
    ax_s, ay_s = frontier_custom_offsets.get(name, (-45, -24))
    annotations_9.append(
        dict(
            x=np.log10(r["total_cost_usd"]),
            y=r["Composite_Capability_Index"],
            text=f"<b>{name}</b> · {fmt_usd_short(r['total_cost_usd'])} <span style='color:#64748B;'>({r['Composite_Capability_Index']:.1f})</span>",
            showarrow=True,
            arrowhead=0,
            arrowcolor=CORAL,
            arrowwidth=1.1,
            ax=ax_s,
            ay=ay_s,
            font=dict(size=11, color=NAVY),
        )
    )

# 2. Well-known non-frontier models labeled cleanly below/right so zero overlap occurs
peer_custom_offsets = {
    "DeepSeek-V3": (58, 12),
    "Claude 2": (55, 18),
    "Mistral Large": (62, 16),
    "Gemini 1.0 Ultra": (68, 22),
    "GPT-4": (56, 36),
    "Grok 4": (48, 22),
    "Grok 3": (-18, 28),
}

for _, r in non_frontier[non_frontier["base_model"].isin(peer_custom_offsets.keys())].iterrows():
    name = r["base_model"]
    ax_s, ay_s = peer_custom_offsets[name]
    annotations_9.append(
        dict(
            x=np.log10(r["total_cost_usd"]),
            y=r["Composite_Capability_Index"],
            text=f"{name} · {fmt_usd_short(r['total_cost_usd'])} ({r['Composite_Capability_Index']:.1f})",
            showarrow=True,
            arrowhead=0,
            arrowcolor="#CBD5E1",
            arrowwidth=1,
            ax=ax_s,
            ay=ay_s,
            font=dict(size=10, color="#64748B"),
        )
    )

# 3. Clean directional guides in opposite corners
annotations_9.extend([
    dict(
        text="↖ <b>Optimal Frontier</b> (Higher Capability, Lower Cost)",
        x=0.02,
        y=0.96,
        xref="paper",
        yref="paper",
        xanchor="left",
        yanchor="top",
        showarrow=False,
        font=dict(size=11, color=CORAL),
    ),
    dict(
        text="Higher Spend / Diminishing Returns ↘",
        x=0.98,
        y=0.04,
        xref="paper",
        yref="paper",
        xanchor="right",
        yanchor="bottom",
        showarrow=False,
        font=dict(size=10.5, color="#94A3B8"),
    ),
])

fig9.update_layout(
    title=dict(
        text=(
            "<b>AI Economics Pareto Frontier: Capability Index vs. Training Cost</b><br>"
            "<span style='font-size:12px; color:#64748B;'>"
            "Models on the dashed frontier deliver the highest capability per dollar; labels show Training Cost and Capability Score (0–100)"
            "</span>"
        ),
        x=0.01,
        xanchor="left",
        y=0.96,
        yanchor="top",
    ),
    font=dict(family="Segoe UI, Arial, sans-serif", size=12, color="#2C3E50"),
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255,255,255,0.9)",
    ),
    height=650,
    margin=dict(t=120, b=65, l=75, r=85),
    annotations=annotations_9,
)

fig9.update_xaxes(
    title_text="Estimated Training Cost (USD — Log Scale)",
    type="log",
    range=[4.7, 8.75],
    tickvals=[1e5, 3e5, 1e6, 3e6, 1e7, 3e7, 1e8, 3e8],
    ticktext=["$100k", "$300k", "$1M", "$3M", "$10M", "$30M", "$100M", "$300M"],
    gridcolor="#F1F5F9",
    linecolor="#CBD5E1",
)
fig9.update_yaxes(
    title_text="Composite Frontier Capability Index (0–100)",
    range=[39, 91],
    gridcolor="#F1F5F9",
    linecolor="#CBD5E1",
)

fig9.show()
fig9.write_image("../assets/charts/chart9_pareto_frontier.png", width=1200, height=650, scale=2)